# Build psychological profiles for the training set

Generates a 30-facet NEO-PI-R profile (~1000 tokens) for every essay in `train.csv`
using `rag.profiler`. Profiles are persisted as JSONL and later embedded into a
trait-aware vector database for the profile-based RAG pipeline.

**This step is offline and one-shot.** Profiles are reusable forever; their quality
bounds the quality of every downstream prediction. **Use a strong model** here
(`gpt-4o`, not `gpt-4o-mini`) — the profiler has to make 30 facet judgments per
essay, and weak profilers propagate the same biases the classifier already has.

Bias-reduction techniques layered into the profiler prompt (see `rag/profiler/README.md`):
label-aware but counterfactual scanning, anti-genre-bias clause for C/N facets, first-class
`n/e` (not-evidenced) value, paraphrase anchoring, no-justification-by-label rule,
closing self-check.

In [1]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "rag").exists():
    project_root = (project_root / ".." / "..").resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.profiler.runner import build_profiles
from rag.profiler.store import ProfileStore
from rag.profiler.prompts import (
    FACETS,
    SIGNAL_VOCAB,
    build_profiler_prompts,
    parse_profile_output,
    slice_profile_for_trait,
)

print("Project root:", project_root)
print("Facets in schema:", len(FACETS))
print("Signal vocab:", SIGNAL_VOCAB)

Project root: F:\std\GR\code\model_x_ocean
Facets in schema: 30
Signal vocab: ('high', 'mod', 'low', 'none', 'n/e')


## Configuration

In [2]:
# Inputs
train_csv  = str(project_root / "data/split/essays/train.csv")

# Outputs
profile_db_dir = str(project_root / "data/profile_db/essays")
log_dir        = str(project_root / "log/profiler")

# Profiler model — use a STRONG model. Profiles are generated once and reused.
model_name   = "gpt-4o-mini"
use_labels   = True       # expose trait labels as loose anchors (prompt forbids label-justified reasoning)

# Optional: limit the number of essays profiled in this run (None = all)
limit        = None

# Operational knobs
checkpoint_every = 25     # save store to disk every N essays
progress_every   = 10     # print progress every N essays

train_df = pd.read_csv(train_csv)
if limit is not None:
    train_df = train_df.head(limit).copy()

print(f"Train  : {len(train_df):>4} rows -> {profile_db_dir}")
print(f"Model  : {model_name}  | use_labels={use_labels}")

Train  : 1974 rows -> F:\std\GR\code\model_x_ocean\data\profile_db\essays
Model  : gpt-4o-mini  | use_labels=True


## Smoke test on 1 essay

Before running the full corpus, render the prompt for one essay and inspect it.
Confirms the labels block, anti-genre clause, and 30-facet reference all assemble correctly.

In [3]:
_row = train_df.iloc[0]
_labels = {col: str(_row[col]).strip().lower() for col in ("cOPN","cCON","cEXT","cAGR","cNEU") if col in _row}
_sys, _usr = build_profiler_prompts(text=str(_row["text"]), trait_labels=_labels)

print("--- SYSTEM PROMPT (first 600 chars) ---")
print(_sys[:600])
print("\n... (truncated; total {} chars)".format(len(_sys)))
print("\n--- USER PROMPT (first 800 chars) ---")
print(_usr[:800])
print("\n... (truncated; total {} chars)".format(len(_usr)))

--- SYSTEM PROMPT (first 600 chars) ---
You are a personality-psychology coding analyst. Your job is to read a
short stream-of-consciousness essay and emit a STRUCTURED psychological
profile keyed to the 30 NEO-PI-R facets plus a short linguistic
fingerprint.

The profile will be stored in a vector database and used as retrieval
context for a separate downstream classifier. Quality of the profile
directly determines the quality of every downstream prediction.

You will be given the writer's five Big-Five trait labels (high/low) as
LOOSE ANCHORS. The labels are weak and noisy. You MUST NOT:
  - justify any evidence by referencing the

... (truncated; total 2907 chars)

--- USER PROMPT (first 800 chars) ---
TRAIT LABELS (loose anchor only; do not justify by these):
  Openness           (cOPN) = high
  Conscientiousness  (cCON) = high
  Extraversion       (cEXT) = low
  Agreeableness      (cAGR) = low
  Neuroticism        (cNEU) = high

FACET REFERENCE (use these codes verbatim, in this e

## Run the profiler over the full training set

Append-safe: re-running this cell after a partial run skips already-profiled
essays. Outputs land in `<profile_db_dir>/profile_store.jsonl`.

In [4]:
store = build_profiles(
    data             = train_df,
    output_dir       = profile_db_dir,
    model_name       = model_name,
    log_dir          = log_dir,
    use_labels       = use_labels,
    checkpoint_every = checkpoint_every,
    progress_every   = progress_every,
)

print(f"\nTotal profiles in store: {len(store)}")

[20260504-091248] Profiling 1974 essays -> F:\std\GR\code\model_x_ocean\data\profile_db\essays\profile_store.jsonl
[20260504-091248] Model: gpt-4o-mini  | use_labels=True
[20260504-091248] Logs : F:\std\GR\code\model_x_ocean\log\profiler\20260504-091248_log.txt
[20260504-091248] Resuming: 1625 essays already profiled.
  [profiler] 1635/1974 done. errors=0 invalid=0 rate=0.06/s ETA=5269s
  [profiler] 1645/1974 done. errors=0 invalid=0 rate=0.07/s ETA=4825s
  [profiler] 1655/1974 done. errors=0 invalid=0 rate=0.07/s ETA=4750s
  [profiler] 1665/1974 done. errors=0 invalid=0 rate=0.07/s ETA=4445s
  [profiler] 1675/1974 done. errors=0 invalid=0 rate=0.07/s ETA=4111s
  [profiler] 1685/1974 done. errors=0 invalid=0 rate=0.07/s ETA=3959s
  [profiler] 1695/1974 done. errors=0 invalid=0 rate=0.07/s ETA=3911s
  [profiler] 1705/1974 done. errors=0 invalid=0 rate=0.07/s ETA=3790s
  [profiler] 1715/1974 done. errors=0 invalid=0 rate=0.07/s ETA=3635s
  [profiler] 1725/1974 done. errors=0 invalid=0 ra

## Validation checks

Three cheap sanity checks (per `rag/profiler/README.md`):
1. parse-validity rate (>95% expected),
2. per-facet class balance (no facet should be >85% one-signal across the corpus),
3. logistic-regression sanity check on facet signals as features (per-trait accuracy >0.55 expected).

If any check fails on the first ~100 essays, **stop and revise the prompt** before profiling the rest.

In [5]:
# Reload the store so this section can be re-run independently of the run above
store = ProfileStore(str(Path(profile_db_dir) / "profile_store.jsonl"))
store.load()
entries = store.get_all()
n = len(entries)
valid_n = sum(1 for e in entries if e.get("valid"))
print(f"Profiles loaded: {n} (valid={valid_n}, parse-rate={valid_n/max(n,1):.1%})")

Profiles loaded: 1974 (valid=1974, parse-rate=100.0%)


In [6]:
# Per-facet class balance
from collections import Counter
import pandas as pd

facet_codes = [c for c, *_ in FACETS]
rows = []
for code in facet_codes:
    c = Counter()
    for e in entries:
        sig = e.get("facets", {}).get(code, {}).get("signal", "")
        if sig:
            c[sig] += 1
    total = sum(c.values()) or 1
    dom = c.most_common(1)[0] if c else ("-", 0)
    rows.append({
        "facet": code,
        "high":  c.get("high", 0),
        "mod":   c.get("mod",  0),
        "low":   c.get("low",  0),
        "none":  c.get("none", 0),
        "n/e":   c.get("n/e",  0),
        "dom_signal": dom[0],
        "dom_share":  dom[1] / total,
    })
balance_df = pd.DataFrame(rows)
skewed = balance_df[balance_df["dom_share"] > 0.85]
print("Facets with >85% mode collapse (concerning):")
display(skewed)
print("\nFull balance table:")
display(balance_df)

Facets with >85% mode collapse (concerning):


,facet,high,mod,low,none,n/e,dom_signal,dom_share



Full balance table:


,facet,high,mod,low,none,n/e,dom_signal,dom_share
0,N1,1009,171,774,18,2,high,0.511145
1,N2,111,75,766,912,110,none,0.462006
2,N3,152,838,181,725,78,mod,0.424519
3,N4,213,1439,90,165,67,mod,0.728977
4,N5,106,884,228,614,142,mod,0.447822
5,N6,888,395,366,284,41,high,0.449848
6,E1,963,490,248,99,174,high,0.487842
7,E2,627,426,732,41,148,low,0.370821
8,E3,57,1124,381,347,65,mod,0.569402
9,E4,318,1354,139,145,18,mod,0.685917


In [7]:
# Logistic-regression sanity check: do facet signals predict trait labels at all?
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

SIG2NUM = {"high": 1.0, "mod": 0.5, "none": 0.0, "low": -0.5, "n/e": 0.0, "": 0.0}

X_rows, label_rows = [], []
for e in entries:
    if not e.get("valid"):
        continue
    feats = [SIG2NUM.get(e["facets"].get(c, {}).get("signal", ""), 0.0) for c in facet_codes]
    X_rows.append(feats)
    label_rows.append(e.get("trait_labels", {}))
X = np.array(X_rows, dtype=float)

results = []
for trait in ("cOPN", "cCON", "cEXT", "cAGR", "cNEU"):
    y = np.array([1 if (lbl.get(trait) == "high") else 0 for lbl in label_rows])
    if len(set(y)) < 2 or len(y) < 20:
        results.append({"trait": trait, "n": len(y), "acc_cv": float("nan")})
        continue
    clf = LogisticRegression(max_iter=2000, C=1.0)
    cv = cross_val_score(clf, X, y, cv=5, scoring="accuracy")
    results.append({"trait": trait, "n": len(y), "acc_cv": float(cv.mean()), "acc_std": float(cv.std())})
lr_df = pd.DataFrame(results)
print("5-fold CV accuracy of logistic regression on facet signals (sanity floor: ~0.55):")
display(lr_df)

5-fold CV accuracy of logistic regression on facet signals (sanity floor: ~0.55):


,trait,n,acc_cv,acc_std
0,cOPN,1974,0.960483,0.005240
1,cCON,1974,0.893108,0.007613
2,cEXT,1974,0.847024,0.021677
3,cAGR,1974,0.917940,0.012426
4,cNEU,1974,0.934649,0.014453


## Inspect a single profile end-to-end

Look at one parsed profile + its trait-sliced inference view to confirm the
schema produces sensible exemplars.

In [8]:
if entries:
    sample = entries[0]
    print("--- USER ID:", sample["user_id"])
    print("--- LABELS:", sample["trait_labels"])
    print("--- VALID :", sample["valid"])
    print("\n--- RAW PROFILER OUTPUT ---\n")
    print(sample["raw"])
    for trait in ("cOPN", "cCON", "cEXT", "cAGR", "cNEU"):
        print(f"\n--- SLICE FOR {trait} ---")
        print(slice_profile_for_trait(sample, trait))

--- USER ID: user_0
--- LABELS: {'cOPN': 'high', 'cCON': 'high', 'cEXT': 'low', 'cAGR': 'low', 'cNEU': 'high'}
--- VALID : True

--- RAW PROFILER OUTPUT ---

[FACETS]
N1 anxiety            | high  | expresses nervousness about upcoming rowing practice and past injuries causing worry.
N2 hostility          | none  | no evidence of anger or resentment toward others in the text.
N3 depression         | mod   | mentions feeling upset about missing a rowing meeting but does not express deep sadness.
N4 self-conscious     | mod   | shows some nervousness about performance and being judged during workouts.
N5 impulsiveness      | none  | no evidence of difficulty resisting urges or acting without thinking.
N6 vulnerability       | high  | feels overwhelmed by physical challenges and past injuries affecting performance.
E1 warmth             | none  | no evidence of affectionate or friendly engagement with others.
E2 gregariousness     | low   | prefers solitary workouts and expresses discomfo